# *Exploratory Data Analysis* (EDA) {#sec-modul-4}

::: {.callout-important title="Sub-CPMK Pertemuan Ini"}
Mahasiswa mampu melakukan *Exploratory Data Analysis* (EDA) secara statistik deskriptif pada data perkotaan, mendeteksi anomali data (*outlier*), menangani nilai kosong (*missing values*), dan menganalisis korelasi kasar antarvariabel perkotaan menggunakan Python.
:::

## Pendahuluan

[\<masukkan narasi yang menceritakan 'big why' mengeksplorasi karakter dan kualitas data perkotaan sebelum menganalisisnya di sini\>]{.teks-placeholder style="color: #8a8a8a;"}

## Gambaran Umum Kasus

Dalam pengerjaan studio, kompilasi data adalah tahap yang pantang dilewati: sebelum satu garis pun digambar ke laporan akhir, seluruh kuesioner kasar hasil survei lapangan diperiksa dulu satu per satu — adakah lembar yang kosong, adakah angka yang tidak masuk akal, dan pola awal apa yang mulai terlihat. Analisis yang dibangun di atas data yang belum diperiksa ibarat rencana tata ruang yang digambar di atas peta dasar yang salah.

Kebiasaan memeriksa itulah yang dalam analitika perkotaan disebut ***Exploratory Data Analysis* (EDA)**: eksplorasi awal untuk mengenali karakter dan menguji kualitas data sebelum data itu dipakai merumuskan apa pun. Menggunakan profil kelurahan Kota Bandar Lampung dari PODES 2021 (BPS) yang tersimpan dalam basis data hasil kerja kita di Modul 3, kita akan mengenali karakter data numerik lewat statistik deskriptif, membaca frekuensi data kategorikal, memburu nilai-nilai ekstrem (*outlier*) dengan metode IQR, menemukan dan menangani data kosong (*missing values*), memeriksa hubungan kasar antarvariabel lewat korelasi, lalu menutupnya dengan sintesis: seperti apa sesungguhnya karakter sosio-demografi kota ini menurut datanya sendiri.

## Instruksi

Buat notebook baru `modul/modul-04.ipynb` di direktori proyek Anda, sambungkan ke *kernel* `analitika-perkotaan`, dan kerjakan seluruh instruksi di dalamnya.

### Mengenal Karakter Data: Statistik Deskriptif Dasar dengan Pandas

Pemeriksaan pertama seorang perencana terhadap data yang baru diterimanya selalu sama: mengenali karakternya secara ringkas. Untuk itu kita butuh datanya lebih dulu — dan kali ini sumbernya bukan lagi berkas CSV, melainkan **lemari arsip `bdl.sqlite` yang Anda tata sendiri di Modul 3**.

Ada satu pekerjaan penataan ulang: laci `kelurahan` yang Anda isi di Modul 3 baru memuat segelintir kolom fasilitas, padahal EDA menuntut **seluruh profil kelurahan** — topografi, kependudukan, hingga infrastruktur. Maka isi ulang laci itu dengan berkas lengkapnya: baca seluruh kolom `bdl_kelurahan.csv`, buang kolom redundan `nama_kec` (normalisasi Modul 3), lalu tulis ulang tabelnya dengan `to_sql`:

In [16]:
import sqlite3
import pandas as pd

path_db = "../dataset-ganjil-2026-2027/bdl.sqlite"
koneksi = sqlite3.connect(path_db)

path_data = "../dataset-ganjil-2026-2027/bdl_kelurahan.csv"
df_csv = pd.read_csv(path_data)
df_csv = df_csv.drop(columns=["nama_kec"])
df_csv.to_sql("kelurahan", koneksi, if_exists="replace", index=False)

126

Angka `126` menandakan seluruh baris kelurahan tersalin. Perhatikan bedanya dengan Modul 3: `pd.read_csv(path_data)` tanpa argumen `usecols` membaca **semua** kolom yang ada — itulah profil lengkap yang kita butuhkan.

::: {.callout-warning title="Penting!"}
### Jika Inspeksi Tidak Menemukan Tabel `kecamatan` {.unnumbered .unlisted}

Langkah di atas menganggap `bdl.sqlite` Anda sehat: berisi tabel `kecamatan` bawaan kelas. Ingat perilaku `sqlite3.connect(...)` dari Modul 3 — *path* yang keliru tidak menimbulkan galat, melainkan diam-diam membuat basis data baru yang kosong. Bila kueri `JOIN` di bawah nanti menghasilkan kolom `nama_kec` yang kosong seluruhnya, berhenti: periksa `path_db` Anda, hapus file kosong yang telanjur terbuat, dan bila perlu ulangi bagian awal Modul 3.
:::

Sekarang tarik profil lengkap itu ke *DataFrame*, sekaligus menjemput nama kecamatan induk dari laci sebelah dengan `LEFT JOIN` — persis pola Modul 3, ditampung dalam string multi-baris:

In [17]:
kueri_profil = """
SELECT
    kel.*,
    kec.nama_kec
FROM
    kelurahan AS kel
LEFT JOIN
    kecamatan AS kec ON kel.kode_kec = kec.kode_kec
"""

df = pd.read_sql(kueri_profil, koneksi)
print(df.shape)

(126, 34)


Satu notasi baru: `kel.*` pada `SELECT` berarti "seluruh kolom dari tabel `kelurahan`" — sehingga kita tidak perlu mengetik puluhan nama kolom satu per satu. Hasilnya 126 baris dan 34 kolom: seluruh profil kelurahan, plus `nama_kec` hasil `JOIN`.

Kini kenali karakternya. Metode `describe()` merangkum kolom-kolom numerik menjadi delapan angka: cacah data terisi (`count`), rata-rata (`mean`), simpangan baku (`std`), nilai terkecil (`min`), kuartil bawah (`25%`), median (`50%`), kuartil atas (`75%`), dan nilai terbesar (`max`). Tampung dulu kolom-kolom yang ingin diperiksa dalam variabel *list* (pola Modul 2) supaya daftar yang sama bisa dipakai ulang sepanjang modul:

In [18]:
kolom_numerik = ["kepadatan_pddk", "jml_tk_negeri", "jml_tk_swasta", "jml_ra_ba_swasta"]

df[kolom_numerik].describe()

,kepadatan_pddk,jml_tk_negeri,jml_tk_swasta,jml_ra_ba_swasta
count,126.000000,126.000000,126.000000,126.000000
mean,9431.825397,0.103175,2.595238,0.269841
std,8428.116609,0.330560,2.075297,0.662271
min,627.000000,0.000000,0.000000,0.000000
25%,3466.750000,0.000000,1.000000,0.000000
50%,6450.500000,0.000000,2.000000,0.000000
75%,12693.500000,0.000000,4.000000,0.000000
max,43907.000000,2.000000,10.000000,4.000000


Bacalah tabel itu seperti perencana membaca rekap survei:

- **`count` = 126 di semua kolom** — tidak ada data kosong pada keempat kolom ini; kelengkapan kolom lain kita periksa belakangan.

- **`kepadatan_pddk`** — rata-ratanya sekitar 9.432 jiwa/km², tetapi simpangan bakunya hampir sama besar (±8.428): sebaran antarkelurahan sangat lebar. Rentangnya membentang dari 627 sampai 43.907 jiwa/km², dan median (6.450) jauh di bawah rata-rata — tanda distribusinya **menceng**: segelintir kelurahan yang luar biasa padat menyeret rata-rata ke atas. Nilai-nilai ekstrem inilah yang akan kita buru pada bagian *outlier*.

- **`jml_tk_negeri`** — median bahkan kuartil atasnya 0: lebih dari tiga perempat kelurahan tidak memiliki satu pun TK negeri; layanan TK di kota ini nyaris sepenuhnya ditopang swasta.

Satu catatan penyaringan kolom: profil PODES juga memuat `jml_ra_ba_negeri`, tetapi seluruh 126 nilainya seragam 0 — kolom seragam tidak menyumbang informasi bagi statistik deskriptif, maka tidak kita ikutkan di `kolom_numerik`. Temuan "kolom seragam" semacam ini tetap layak dicatat; ia akan kembali di bagian sintesis.

### Data Kategorikal: Melihat Frekuensi dan Modus dengan `value_counts()`

Tidak semua kolom angka bermakna besaran. Pada profil PODES, banyak kolom berisi **kode kategori** — angkanya adalah label, bukan jumlah, sehingga merata-ratakannya tidak bermakna. Perlakuan yang tepat untuk data kategorikal adalah menghitung **frekuensi** tiap kategori dengan `value_counts()` (Modul 2), dan kategori terbanyaknya disebut **modus**. Periksa dua kolom: topografi wilayah (`topo`) dan status daerah saat pencacahan (`status_daerah`):

In [19]:
print(df["topo"].value_counts())
print(df["status_daerah"].value_counts())

topo
3    77
2    49
Name: count, dtype: int64
status_daerah
1    124
2      2
Name: count, dtype: int64


Arti tiap kode tercantum dalam kamus data PODES 2021 yang dibagikan dosen. Untuk `topo`: kode 3 adalah **dataran** dan kode 2 adalah **lembah/daerah aliran sungai** (kode 1, lereng/puncak, tidak muncul di data). Untuk `status_daerah`: kode 1 **perkotaan**, kode 2 **perdesaan**.

Terbaca dua hal. Pertama, modus topografi Bandar Lampung adalah dataran (77 kelurahan), tetapi porsi kelurahan lembah/DAS-nya besar juga (49 kelurahan) — hampir empat dari sepuluh. Kedua, yang lebih menarik: dua kelurahan di kota ini masih tercacah **perdesaan**. Temuan kecil semacam inilah yang sering luput bila kita tidak menyempatkan EDA. (Pandas juga menyediakan metode `mode()` untuk mengambil modus secara langsung; pada keluaran `value_counts()` modus selalu baris teratas, karena hasilnya terurut dari frekuensi terbesar.)

### Menemukan Keanehan: Deteksi *Outlier* (Nilai Ekstrem) Menggunakan Metode IQR

Statistik deskriptif tadi sudah membisikkan adanya nilai-nilai ekstrem. Sekarang kita tangkap mereka secara tegas dengan **metode IQR** (*Interquartile Range*), resep klasik John Tukey yang hanya membutuhkan dua angka dari tabel `describe()` tadi: kuartil bawah (Q1) dan kuartil atas (Q3). Jarak keduanya, `IQR = Q3 - Q1`, adalah rentang "separuh data yang di tengah". Sebuah nilai dicap *outlier* bila jatuh di luar **pagar**: lebih kecil dari `Q1 - 1,5 × IQR`, atau lebih besar dari `Q3 + 1,5 × IQR`.

Hitung pagarnya untuk kepadatan penduduk — metode `quantile()` menerima posisi kuartil dalam pecahan (0.25 untuk Q1, 0.75 untuk Q3):

In [20]:
q1 = df["kepadatan_pddk"].quantile(0.25)
q3 = df["kepadatan_pddk"].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f"Q1 = {q1}, Q3 = {q3}, IQR = {iqr}")
print(f"Pagar bawah = {batas_bawah}, pagar atas = {batas_atas}")

Q1 = 3466.75, Q3 = 12693.5, IQR = 9226.75
Pagar bawah = -10373.375, pagar atas = 26533.625


Pagar bawahnya negatif — dan karena kepadatan penduduk mustahil negatif, otomatis tidak ada *outlier* di sisi bawah. Perburuan tinggal ke sisi atas: kelurahan mana yang kepadatannya menembus 26.533 jiwa/km²? Saring dengan filtrasi baris Modul 2:

In [21]:
outlier_kepadatan = df[(df["kepadatan_pddk"] < batas_bawah) | (df["kepadatan_pddk"] > batas_atas)]
outlier_kepadatan[["nama_des", "nama_kec", "kepadatan_pddk"]]

,nama_des,nama_kec,kepadatan_pddk
7,Kota Karang,Telukbetung Timur,43907
31,Tanjung Agung,Tanjung Karang Timur,37717
32,Kebon Jeruk,Tanjung Karang Timur,26592
33,Sawah Lama,Tanjung Karang Timur,35775
63,Sukajawa,Tanjung Karang Barat,31830
82,Sukamenanti,Kedaton,28790


Enam kelurahan tertangkap pagar, dipuncaki Kota Karang (Telukbetung Timur) dengan 43.907 jiwa/km² — hampir tujuh kali lipat median kota. Ingat: *outlier* bukan otomatis data yang salah. Kepadatan setinggi itu bisa jadi kenyataan permukiman pesisir yang memang sesak — justru karena itulah ia perlu ditandai: diperiksa kebenarannya, dan bila benar, diberi perhatian khusus dalam perencanaan.

Satu kolom sudah; tetapi memeriksa kolom-kolom lain dengan menyalin-tempel blok kode yang sama berulang kali jelas bukan cara kerja yang tertib. Serahkan pengulangannya kepada Python — jalankan pemeriksaan IQR untuk **setiap** kolom dalam `kolom_numerik` sekaligus, lengkap dengan penilaian otomatis atas hasilnya:

In [22]:
for kolom in kolom_numerik:
    q1 = df[kolom].quantile(0.25)
    q3 = df[kolom].quantile(0.75)
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr

    penanda_outlier = (df[kolom] < batas_bawah) | (df[kolom] > batas_atas)
    jumlah_outlier = penanda_outlier.sum()

    if jumlah_outlier == 0:
        keterangan = "aman"
    elif jumlah_outlier <= 5:
        keterangan = "perlu dicermati"
    else:
        keterangan = "perlu investigasi"

    print(f"{kolom}: {jumlah_outlier} kelurahan outlier - {keterangan}")

kepadatan_pddk: 6 kelurahan outlier - perlu investigasi
jml_tk_negeri: 12 kelurahan outlier - perlu investigasi
jml_tk_swasta: 1 kelurahan outlier - perlu dicermati
jml_ra_ba_swasta: 22 kelurahan outlier - perlu investigasi


Empat kolom terperiksa oleh satu blok kode. Dua hal kecil pada kode itu: `penanda_outlier` adalah deretan nilai benar/salah hasil filtrasi Modul 2, dan `.sum()` menghitung banyaknya `True` di dalamnya (Python menghitung `True` sebagai 1). Hasilnya patut direnungkan: `jml_ra_ba_swasta` dilaporkan punya 22 "*outlier*" dan `jml_tk_negeri` 12 — padahal kita tahu dari `describe()` bahwa mayoritas nilai kedua kolom itu 0, sehingga Q1 dan Q3-nya sama-sama 0, IQR-nya 0, dan **setiap nilai di atas nol otomatis dicap *outlier***. Pelajarannya: keluaran metode adalah alarm awal, bukan vonis — penilaian akhir tetap di tangan perencana yang memahami konteks datanya.

::: {.callout-note title="Konsep Python"}
### Perulangan `for` dan `range()` {.unnumbered .unlisted}

Baris `for kolom in kolom_numerik:` memerintahkan Python: "untuk setiap isi *list* `kolom_numerik`, jalankan blok di bawah ini satu kali." Pada setiap putaran, variabel `kolom` bergiliran menampung satu isi *list* — mula-mula `"kepadatan_pddk"`, lalu `"jml_tk_negeri"`, dan seterusnya sampai isi *list* habis. Yang termasuk "blok di bawah ini" adalah semua baris yang **berindentasi** (menjorok) di bawah baris `for` — begitu indentasi berakhir, berakhir pula wilayah pengulangan.

Ini persis ritme kompilasi data di studio: satu daftar periksa yang sama diterapkan ke setiap kuesioner dalam tumpukan, kuesioner demi kuesioner, sampai tumpukan habis — tanpa menulis ulang daftar periksanya untuk tiap lembar.

Bila yang dibutuhkan bukan "untuk setiap isi *list*" melainkan "ulangi sekian kali", pasangkan `for` dengan `range()`: `for i in range(4):` mengulang blok empat putaran, dengan `i` bernilai 0, 1, 2, 3 secara bergiliran (ingat, Python menghitung dari nol). Bentuk inilah yang akan sering muncul ketika kita menjelajah struktur data yang lebih dalam di modul-modul lanjut.
:::

::: {.callout-note title="Konsep Python"}
### Percabangan `if`–`elif`–`else` {.unnumbered .unlisted}

Blok `if`–`elif`–`else` membuat kode **memilih jalan**: Python memeriksa kondisi satu per satu dari atas — kondisi `if` dulu, lalu tiap `elif` (singkatan *else if*) — dan begitu menemukan kondisi pertama yang benar, hanya blok cabang itulah yang dijalankan; sisanya dilewati. Cabang `else` adalah penampung terakhir bila tidak ada satu pun kondisi yang benar. Kondisinya sendiri dibangun dari perkakas yang sudah Anda kuasai di Modul 2: operator perbandingan (`==`, `<=`, `>`) dan operator boolean.

Analoginya adalah momen menyortir kuesioner di studio: tiap lembar hasil survei Anda periksa dengan urutan pertanyaan yang tetap — terisi lengkap? masukkan ke tumpukan siap-olah; kurang satu-dua isian? beri catatan perbaikan; rusak parah? sisihkan untuk survei ulang. Satu lembar hanya berakhir di satu tumpukan, sebagaimana satu putaran percabangan hanya menjalankan satu cabang.

Perhatikan pula bahwa percabangan tadi bekerja **di dalam** perulangan: penilaian yang sama diterapkan pada setiap kolom yang digilir `for`. Memadukan keduanya — mengulang sambil memilih — adalah pola alur kendali yang akan terus Anda pakai sampai akhir buku ini.
:::

### Mengisi Lembar Kosong: Identifikasi dan Penanganan *Missing Values*

Setelah nilai yang "terlalu ada", giliran nilai yang **tidak ada**. Dalam *DataFrame*, isian yang kosong ditandai `NaN` (*Not a Number*), dan metode `isna()` memeriksanya sel demi sel. Rangkaikan dengan `.sum()` untuk merekap jumlah kekosongan per kolom, lalu — karena kolomnya 34 — saring rekap itu agar hanya kolom yang benar-benar bermasalah yang tampil:

In [23]:
jumlah_kosong = df.isna().sum()
jumlah_kosong[jumlah_kosong > 0]

dm_permukiman_sutet    119
jml_bangunan_sutet     123
dtype: int64

Dari 34 kolom, hanya dua yang berlubang — tetapi lubangnya menganga: `dm_permukiman_sutet` kosong pada 119 baris dan `jml_bangunan_sutet` pada 123 baris, dari total 126. Jembatan kecil ke Modul 3: `NaN` di Pandas adalah kembaran `NULL` di SQL — kekosongan yang sama yang dulu kita saring dengan `IS NULL`. Buktikan kesetaraannya dengan bertanya langsung ke basis data:

In [24]:
kueri_kosong = """
SELECT
    COUNT(*) AS jumlah_baris_kosong
FROM
    kelurahan
WHERE
    jml_bangunan_sutet IS NULL
"""

pd.read_sql(kueri_kosong, koneksi)

,jumlah_baris_kosong
0,123


Angka yang sama, 123 — dua kacamata, satu kenyataan.

Sebelum menambal, **diagnosis dulu penyebab kosongnya**. Ketiga kolom ini berkaitan: `dm_dilalui_sutet` mencatat apakah wilayah kelurahan dilalui jaringan listrik tegangan tinggi (SUTET/SUTT/SUTTAS; kode 1 ada, kode 2 tidak — lihat kamus data PODES 2021), `dm_permukiman_sutet` mencatat ada-tidaknya permukiman di bawah jaringan itu, dan `jml_bangunan_sutet` menghitung bangunannya. Pertanyaan lanjutan tentu hanya diisi bila jawaban pertanyaan awalnya "ada". Buktikan dengan filtrasi silang:

In [25]:
df[df["dm_dilalui_sutet"] == 1][["nama_des", "dm_permukiman_sutet", "jml_bangunan_sutet"]]

,nama_des,dm_permukiman_sutet,jml_bangunan_sutet
4,Batu Putuk,2.0,NaN
11,Gedong Pakuon,2.0,NaN
22,Srengsem,2.0,NaN
68,Sumber Agung,1.0,12.0
109,Korpri Jaya,1.0,4.0
111,Korpri Raya,1.0,25.0
112,Sukarame Baru,2.0,NaN


Tepat 7 kelurahan dilalui SUTET — dan hanya pada merekalah `dm_permukiman_sutet` terisi; `jml_bangunan_sutet` bahkan hanya terisi pada 3 kelurahan yang permukimannya memang ada di bawah jaringan (kode 1). Kekosongan seperti ini disebut **kosong struktural**: bukan petugas lupa mencatat, melainkan pertanyaannya memang tidak berlaku bagi 119 kelurahan lain. Diagnosis ini menentukan resep penanganannya.

Pandas menyediakan dua penanganan pabrikan: `dropna()` membuang setiap baris yang memuat kekosongan, dan `fillna(nilai)` menambal kekosongan dengan nilai tertentu. Coba dulu yang pertama — tetapi periksa dampaknya lewat `.shape` sebelum benar-benar memakainya:

In [26]:
print(df.shape)
print(df.dropna().shape)

(126, 34)
(3, 34)


::: {.callout-warning title="Penting!"}
### `dropna()` Bukan Jawaban Otomatis {.unnumbered .unlisted}

Membuang semua baris yang berlubang menyisakan **3 kelurahan dari 126** — kita kehilangan 97% kota hanya karena dua kolom yang kosongnya pun struktural. Baris di atas sengaja tidak menimpa `df`: `df.dropna()` menghasilkan *DataFrame* baru, dan selama hasilnya tidak ditugaskan kembali ke variabel, data asli tetap utuh. Jadikan `.shape` kebiasaan wajib sebelum memutuskan membuang baris — dan buang hanya bila kekosongannya sedikit serta barisnya memang tidak terpakai.
:::

Untuk kasus kita, resep yang masuk akal justru menambal. Pada `jml_bangunan_sutet`, kosong struktural bermakna pasti: tidak dilalui SUTET, atau tidak ada permukiman di bawahnya — artinya **nol bangunan**. Tambal dengan `fillna(0)`:

In [27]:
df["jml_bangunan_sutet"] = df["jml_bangunan_sutet"].fillna(0)

print(df["jml_bangunan_sutet"].isna().sum())

0


Kekosongannya tuntas. Adapun `dm_permukiman_sutet` kita **biarkan kosong**: ia kolom kode kategori (1/2), dan menambalnya dengan angka karangan justru memalsukan jawaban "pertanyaan ini tidak berlaku". Tidak semua lubang harus ditambal — yang wajib adalah *mengetahui* lubangnya dan *sadar* memutuskan perlakuannya.

Rumuskan keputusan itu secara tertib untuk ketiga kolom SUTET sekaligus — perulangan dan percabangan yang baru Anda pelajari kini bekerja berpasangan:

In [28]:
kolom_sutet = ["dm_dilalui_sutet", "dm_permukiman_sutet", "jml_bangunan_sutet"]

for kolom in kolom_sutet:
    persen_kosong = df[kolom].isna().sum() / len(df) * 100

    if persen_kosong == 0:
        keputusan = "lengkap, siap dianalisis"
    elif persen_kosong < 50:
        keputusan = "tambal dengan fillna"
    else:
        keputusan = "periksa: kemungkinan kosong struktural"

    print(f"{kolom}: {persen_kosong:.1f}% kosong - {keputusan}")

dm_dilalui_sutet: 0.0% kosong - lengkap, siap dianalisis
dm_permukiman_sutet: 94.4% kosong - periksa: kemungkinan kosong struktural
jml_bangunan_sutet: 0.0% kosong - lengkap, siap dianalisis


Perhatikan `jml_bangunan_sutet` kini dilaporkan lengkap — buah dari tambalan barusan. (Notasi `:.1f` di dalam f-string Modul 1 sekadar merapikan tampilan angka menjadi satu digit di belakang koma.)

### Hubungan Awal: Analisis Korelasi Kasar Antarvariabel Numerik

Karakter tiap kolom sudah dikenali satu per satu; pemeriksaan pamungkas EDA adalah melihat **hubungan antar**-kolom. Ukuran kasarnya adalah **koefisien korelasi** (r): angka antara −1 dan +1 yang menunjukkan arah dan kekuatan hubungan garis lurus dua variabel — mendekati +1 berarti keduanya cenderung naik bersama, mendekati −1 berarti yang satu naik ketika yang lain turun, dan di sekitar 0 berarti nyaris tak ada pola bersama. Metode `corr()` menghitungnya untuk semua pasangan kolom sekaligus. Ujilah dugaan yang wajar bagi perencana: *makin padat sebuah kelurahan, makin banyak pula fasilitas pendidikannya?*

In [29]:
kolom_korelasi = ["kepadatan_pddk", "jml_tk_negeri", "jml_tk_swasta", "jml_sd_negeri", "jml_sd_swasta"]

df[kolom_korelasi].corr()

,kepadatan_pddk,jml_tk_negeri,jml_tk_swasta,jml_sd_negeri,jml_sd_swasta
kepadatan_pddk,1.000000,-0.073817,-0.192438,0.155989,-0.041028
jml_tk_negeri,-0.073817,1.000000,0.003054,-0.061323,-0.102833
jml_tk_swasta,-0.192438,0.003054,1.000000,0.034594,0.536261
jml_sd_negeri,0.155989,-0.061323,0.034594,1.000000,-0.074743
jml_sd_swasta,-0.041028,-0.102833,0.536261,-0.074743,1.000000


Cara membaca **matriks korelasi** ini: tiap sel adalah r untuk pasangan kolom baris-kolomnya; diagonalnya selalu 1 (setiap variabel berkorelasi sempurna dengan dirinya sendiri); dan matriksnya simetris — cukup baca separuhnya.

Hasilnya menampar dugaan kita. Korelasi kepadatan penduduk dengan keempat kolom fasilitas semuanya lemah (dari −0,192 sampai 0,156): pada data ini, **kelurahan yang lebih padat tidak serta-merta lebih kaya fasilitas pendidikan** — sinyal awal yang layak menjadi bahan diskusi pemerataan layanan. Justru pasangan terkuat muncul di tempat lain: `jml_tk_swasta` dengan `jml_sd_swasta` (r = 0,536, kekuatan sedang) — kelurahan yang banyak TK swastanya cenderung banyak pula SD swastanya, gelagat bahwa penyedia layanan swasta berkerumun di kantong pasar yang sama.

::: {.callout-warning title="Penting!"}
### Korelasi Bukan Sebab-Akibat {.unnumbered .unlisted}

Sekuat apa pun r, korelasi hanya mencatat dua hal bergerak bersama — ia tidak pernah membuktikan yang satu *menyebabkan* yang lain. Analisis kita di sini juga sengaja disebut korelasi **kasar**: belum ada uji signifikansi atau kendali atas variabel lain; perkakas statistika yang lebih ketat untuk itu Anda pelajari di mata kuliah statistika. Dalam EDA, posisi korelasi adalah pemandu — penunjuk hubungan mana yang pantas diselidiki lebih dalam, bukan hakim yang memutus perkara.
:::

### Sintesis EDA: Menarik Kesimpulan Karakteristik Sosio-Demografi Kota

Pemeriksaan selesai; saatnya menyusun lembar kompilasi versi perencana. Digabungkan, temuan-temuan EDA kita membentuk potret karakter sosio-demografi Kota Bandar Lampung menurut datanya sendiri:

- **Kepadatan yang timpang.** Separuh kelurahan berkepadatan di bawah 6.450 jiwa/km², tetapi ekor distribusinya panjang ke atas hingga 43.907 — enam kelurahan ekstrem (terpadat: Kota Karang) menyeret rata-rata kota jauh di atas mediannya.

- **Kota di atas dataran dan lembah.** Modus topografi adalah dataran (77 kelurahan), disusul lembah/DAS (49); dan dua kelurahan ternyata masih tercacah berstatus perdesaan.

- **Layanan pendidikan usia dini bertumpu pada swasta.** Lebih dari tiga perempat kelurahan tanpa TK negeri, dan RA/BA negeri tidak ada sama sekali (kolom seragam nol).

- **Kekosongan data yang bermakna.** Dua kolom SUTET kosong secara struktural — kosongnya adalah informasi ("tidak berlaku"), bukan kelalaian; satu kolom kita tambal nol, satu lagi sengaja dibiarkan.

- **Kepadatan tidak menjamin fasilitas.** Korelasi kepadatan–fasilitas pendidikan lemah; yang menonjol justru kerumunan sesama layanan swasta (TK–SD swasta, r = 0,536).

Potret inilah bekal kita melangkah: dugaan mana yang pantas diuji serius, kolom mana yang siap pakai, dan angka mana yang harus diperlakukan hati-hati. Alur perkakasnya pun kini utuh — basis data menertibkan penyimpanan (Modul 3), Pandas mengolah (Modul 2), EDA mengenali karakter dan kualitasnya (modul ini) — dan pada Modul 5, temuan-temuan yang masih berupa angka ini akan kita bunyikan menjadi **visualisasi**.

Tutup sesi kerja basis data Anda:

In [30]:
koneksi.close()

## Latihan Mandiri

Kerjakan di notebook baru `modul/latihan-04.ipynb` dengan *kernel* `analitika-perkotaan`. Alurnya sama persis dengan instruksi di atas; yang berganti adalah fasilitasnya — dari TK ke **pendidikan dasar (SD dan SMP)** — plus satu kolom kategorikal yang lebih kaya.

1. **Buka lemari arsip.** Buka koneksi ke `bdl.sqlite`, lalu tarik profil lengkap kelurahan ber-`nama_kec` ke *DataFrame* memakai `read_sql` dengan kueri `LEFT JOIN` seperti pada instruksi.

2. **Kolom total.** Buat dua kolom baru (Modul 2): `jml_sd` dari `jml_sd_negeri + jml_sd_swasta`, dan `jml_smp` dari `jml_smp_negeri + jml_smp_swasta`.

3. **Karakter data.** Tampung `["jml_sd", "jml_smp"]` dalam variabel *list*, tampilkan `describe()`-nya, lalu tuliskan di sel teks: berapa median masing-masing, dan apakah sebarannya lebar?

4. **Perburuan outlier.** Dengan perulangan `for` atas *list* tadi, hitung pagar IQR tiap kolom, cacah *outlier*-nya, dan berikan keterangan otomatis memakai percabangan `if`–`elif`–`else` seperti pada instruksi. Kelurahan mana pemuncak *outlier* `jml_sd`?

5. **Frekuensi lapangan usaha.** Rekap `value_counts()` untuk kolom kategorikal `lap_usaha_sebagian_warga` — sumber penghasilan utama sebagian besar warga tiap kelurahan. Arti kode yang muncul di data Bandar Lampung (selengkapnya lihat kamus data PODES 2021 yang dibagikan dosen):

   | Kode | Lapangan usaha |
   |---:|---|
   | 1 | Pertanian, kehutanan, dan perikanan |
   | 3 | Industri pengolahan |
   | 5 | Pengadaan air, pengelolaan sampah, limbah, dan daur ulang |
   | 6 | Konstruksi |
   | 7 | Perdagangan besar dan eceran; reparasi mobil dan sepeda motor |
   | 8 | Transportasi dan pergudangan |
   | 9 | Penyediaan akomodasi dan makan minum |
   | 13 | Jasa perusahaan |
   | 14 | Administrasi pemerintahan, pertahanan, dan jaminan sosial wajib |
   | 17 | Jasa lainnya |

   Tuliskan satu kalimat: lapangan usaha apa yang menjadi tumpuan terbanyak kelurahan-kelurahan Bandar Lampung, dan masuk akalkah itu untuk sebuah kota?

6. **Kelengkapan dan hubungan.** Periksa kekosongan kolom-kolom yang Anda pakai dengan `isna().sum()`, lalu hitung matriks korelasi `df[["jml_sd", "jml_smp"]].corr()`. Seberapa kuat hubungan jumlah SD dengan jumlah SMP antarkelurahan?

7. **Sintesis dan tutup.** Tutup notebook dengan satu paragraf sintesis ala laporan kompilasi: karakter sebaran fasilitas pendidikan dasar, temuan *outlier*-nya, kelengkapan datanya, dan makna korelasi SD–SMP bagi pemerataan layanan. Tutup koneksi basis data, lalu jalankan **Restart *kernel* & Run All** — seluruh sel harus lolos berurutan.